## Cosmic Web - Synthesis

In this notebook we read what the notebooks recorded and draw the comparison: the same histogram from every tool, one call per tool on a shared timeline, each tool's time against its bytes roof, and the curve of the simulation cube.

### Table of Contents

1. [Stage files](#1-stage-files)
2. [One call per tool](#2-one-call-per-tool)
3. [Bytes roof](#3-bytes-roof)
4. [The simulation cube](#4-the-simulation-cube)
5. [Four programming models](#5-four-programming-models)

---

### 1. Stage files

Each notebook left one file per stage under `timings/`: its histogram, timing, launch count and trace. The four compared are the CuPy reference, JAX, cuda.compute and the CppJIT kernel. The TODO versions appear in the table and in §3.

In [ ]:
import numpy as np

import cosmic_core as cw

STAGES = ['cupy', 'jax', 'cccl', 'cppjit_cuda']
EXTRAS = ['cupy_views', 'jax_private', 'cppjit_cub']
LABELS = {'cupy': 'CuPy', 'cupy_views': 'CuPy views', 'jax': 'JAX', 'jax_private': 'JAX slabs',
          'cccl': 'cuda.compute', 'cppjit_cub': 'CppJIT CUB', 'cppjit_cuda': 'CppJIT CUDA'}

by_stage = cw.load_stages(STAGES, EXTRAS)
N, K = by_stage['cupy']['n'], by_stage['cupy']['k']

print(f"\n{'tool':<12} {'N':>5} {'K':>4} {'warm ms':>9} {'cold ms':>9} {'max|Δχ|':>8} {'Σ':>3}")
for s, r in by_stage.items():
    cold = f"{r['cold_s'] * 1e3:9.1f}" if 'cold_s' in r else f"{'-':>9}"
    print(f"{LABELS[s]:<12} {r['n']:>4}^3 {r['k']:>4} {r['median_s'] * 1e3:9.2f} {cold} {r.get('max_abs_diff', '?'):>8} {r.get('hist_sum', '?'):>3}")

### 2. One call per tool

The same histogram from each tool, in three figures: the curves with every tool's difference to the reference underneath, one call per tool on a shared time axis, and where its wall time goes. Does any tool wait on the host?

In [ ]:
edges = cw.bin_edges(K)
reference = cw.chi_reference(N)
curves = {LABELS[s]: cw.ecc_from_hist(by_stage[s]['hist']) for s in STAGES if s in by_stage and 'hist' in by_stage[s]}
cw.plot_curves_with_delta(edges, curves, reference, title=f'χ(ν) at N = {N}^3 from every stage file')

traced = [s for s in STAGES if s in by_stage and 'trace' in by_stage[s] and 'launches' in by_stage[s]]
cw.plot_timelines({LABELS[s]: by_stage[s]['trace'] for s in traced},
                           title='one call per tool: GPU operations on a shared time axis')

parts = {LABELS[s]: cw.wall_breakdown(by_stage[s]) for s in traced}
cw.plot_breakdown(parts, title='where one call spends its wall time')

- Every Δ strip is zero.
- On the timeline, JAX's few long kernels outlast CuPy's many short ones: the launch count does not set the time at this size.
- The host share is small in every bar: no tool waits on the host.

### 3. Bytes roof

Time per call against size for every tool, with each tool's bytes model drawn dashed at the measured copy bandwidth, then the first call against the warm median. The gap between a line and its roof is what the model leaves out: launches, synchronisation, atomics and the corner reads. Which tools reach their roofs?

In [ ]:
gbs = cw.copy_roof_gbs()
swept = [s for s in by_stage if 'sweep' in by_stage[s]]
cw.sweep_series(by_stage, {s: LABELS[s] for s in swept})
cw.plot_bytes_roof({LABELS[s]: by_stage[s]['sweep'] for s in swept}, {LABELS[s]: s for s in swept}, gbs,
                   title=f'time per call against N; dashed: bytes model at {gbs:.0f} GB/s')

cold = [s for s in by_stage if 'cold_s' in by_stage[s]]
cw.plot_cold_warm([LABELS[s] for s in cold], [by_stage[s]['cold_s'] for s in cold],
                           [by_stage[s]['median_s'] for s in cold], 'first call against the warm median')

- At the smallest size every tool is far from its roof: the launch count sets the time.
- As the size grows every line runs parallel to its roof. CuPy reaches its roof. The kernel, the library passes and JAX stay above theirs, JAX furthest: its scatters wait on atomic adds.
- The cold bars show each tool's first-use cost, explained in its own notebook.

### 4. The simulation cube

The same histogram on the real cube, or on its Zel'dovich stand-in when the cube is absent, against the Gaussian expectation with $\lambda$ taken from the cube. Does the cube follow it?

In [ ]:
nu, lam, _ = cw.load_dataset(N)
q = cw.quantise(nu, K)
hist = cw.ecc_hist_reference(q, K)
assert hist.sum() == 0, hist.sum()
chi = cw.ecc_from_hist(hist)
pred = cw.chi_gaussian(edges, lam, q.size)
_, zeros = cw.plot_overlay(edges, chi, pred, title=f'χ(ν) of the cube at {q.shape[0]}^3', label='the cube')
print(f'zero crossings: the cube {np.round(zeros, 2).tolist()}, '
      f'the Gaussian expectation {np.round(cw.zero_crossings(edges, pred), 2).tolist()}')

- The printed crossings show where the cube's curve departs from the expectation's $\pm 1$.

### 5. Four programming models

| model | it decides for us | it leaves to us | where it stops |
|---|---|---|---|
| **Array model** (CuPy) | one kernel per expression, every temporary | the expressions, the dtype, views and buffers | nothing fuses across lines |
| **Traced graph** (JAX) | the fusion, the executable, its memory plan | the function and its static arguments | a scatter is a boundary |
| **Algorithm over iterators** (cuda.compute) | the kernel of each algorithm, the fusion of the iterators into it | the algorithm and the iterators | one output per call; an op reads no array |
| **Kernel** (CppJIT CUDA) | the binding and the launcher | the thread mapping, the shared memory, the launch, the stream | the first use pays the compile; a name is defined once per session |

Takeaways:

- The same algorithm had four cost structures: temporaries, fusions, library passes, one kernel.
- The exact integer gate made the four comparable.